# 05 Inference and Reporting Demo

In [ ]:

import os, json, joblib
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from src.preprocessing import clean_column_names, basic_log_preprocess, add_text_length_features, fill_numeric, encode_categoricals, keep_or_create_columns, create_sequences
from src.feature_engineering import extract_url_features
from src.reporting import build_dashboard_summary, write_dashboard_json

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()


In [ ]:

def run_log_inference(source, csv_path):
    model_dir = os.path.join(BASE_DIR, 'models', f'{source}_logs' if source != 'windows' else 'windows_logs')
    model = load_model(os.path.join(model_dir, f'{source}_log_lstm_autoencoder.keras'))
    scaler = joblib.load(os.path.join(model_dir, f'{source}_log_scaler.joblib'))
    cfg = json.load(open(os.path.join(model_dir, f'{source}_log_model_config.json'), 'r', encoding='utf-8'))
    df = pd.read_csv(csv_path)
    df = clean_column_names(df)
    df = basic_log_preprocess(df, cfg['timestamp_col'])
    df = add_text_length_features(df, cfg.get('text_cols', []))
    df = fill_numeric(df)
    df, _ = encode_categoricals(df)
    df = keep_or_create_columns(df, cfg['feature_cols'])
    X = scaler.transform(df[cfg['feature_cols']])
    X_seq = create_sequences(X, cfg['sequence_length'])
    recon = model.predict(X_seq, verbose=0)
    mse = np.mean(np.power(X_seq - recon, 2), axis=(1,2))
    results = []
    for idx, err in enumerate(mse):
        score = min(100.0, round((float(err) / max(cfg['threshold'], 1e-8)) * 50, 2))
        results.append({'source_type': source, 'row_index': idx + cfg['sequence_length'] - 1, 'prediction': 'anomaly' if err > cfg['threshold'] else 'normal', 'threat_score': score})
    return results


In [ ]:

network_results = []
web_results = []
windows_results = []
url_results = []
# شغل السطور دي بعد التدريب ووضع البيانات
# network_results = run_log_inference('network', os.path.join(BASE_DIR, 'data/logs/network/network_logs.csv'))
# web_results = run_log_inference('web', os.path.join(BASE_DIR, 'data/logs/web/web_logs.csv'))
# windows_results = run_log_inference('windows', os.path.join(BASE_DIR, 'data/logs/windows/windows_logs.csv'))


In [ ]:

summary = build_dashboard_summary(url_results=url_results, log_results=network_results + web_results + windows_results)
write_dashboard_json(summary, os.path.join(BASE_DIR, 'reports', 'dashboard_summary.json'))
summary
